# Distributed Training

**Interview Focus**: DDP, FSDP, AllReduce, parallelism strategies, communication overhead.

**Key Concepts**:
- Data Parallelism (DP) vs Distributed Data Parallelism (DDP)
- Model Parallelism vs Pipeline Parallelism vs Tensor Parallelism
- FSDP (Fully Sharded Data Parallel)
- Gradient synchronization via AllReduce
- DeepSpeed ZeRO stages

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. Why Distributed Training?

**Two reasons to go multi-GPU**:
1. **Speed**: Process more data per second (data parallelism)
2. **Size**: Model doesn't fit on one GPU (model parallelism)

| Strategy | What's split | When to use |
|----------|-------------|-------------|
| Data Parallelism | Data across GPUs, full model on each | Model fits on 1 GPU |
| Model Parallelism | Model layers across GPUs | Model too large for 1 GPU |
| Pipeline Parallelism | Model stages across GPUs + micro-batching | Large model, want high utilization |
| Tensor Parallelism | Individual layers split across GPUs | Very large layers (LLMs) |
| FSDP | Parameters, gradients, optimizer states sharded | Medium-large models |

## 2. Data Parallelism: DP vs DDP

### `nn.DataParallel` (DP) — Do NOT Use
- Single process, GIL-limited
- Scatters data, gathers outputs on GPU 0 → GPU 0 bottleneck
- Uneven memory: GPU 0 holds all gradients

### `DistributedDataParallel` (DDP) — The Standard
- One process per GPU, no GIL issues
- AllReduce for gradient synchronization (no single bottleneck)
- Overlaps communication with backward pass
- Even memory distribution

In [ ]:
# Visualize DP vs DDP
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# DP: GPU 0 bottleneck
ax = axes[0]
ax.set_title('DataParallel (DP) — GPU 0 Bottleneck', fontsize=12, fontweight='bold')
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')

# GPU boxes
for i, y_pos in enumerate([6, 4, 2, 0]):
    color = '#FF5722' if i == 0 else '#2196F3'
    ax.add_patch(mpatches.FancyBboxPatch((0.5, y_pos), 3, 1.5, 
                 boxstyle="round,pad=0.1", facecolor=color, alpha=0.3))
    ax.text(2, y_pos + 0.75, f'GPU {i}', ha='center', va='center', fontsize=10)

# Arrows showing gather to GPU 0
for y_pos in [4.75, 2.75, 0.75]:
    ax.annotate('', xy=(5, 6.75), xytext=(4, y_pos),
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

ax.add_patch(mpatches.FancyBboxPatch((5, 5.5), 4.5, 2, 
             boxstyle="round,pad=0.1", facecolor='#FF5722', alpha=0.2))
ax.text(7.25, 6.5, 'GPU 0: gather outputs\ncompute loss\nscatter gradients', 
        ha='center', va='center', fontsize=8)

# DDP: AllReduce
ax = axes[1]
ax.set_title('DistributedDataParallel (DDP) — AllReduce', fontsize=12, fontweight='bold')
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')

for i, y_pos in enumerate([6, 4, 2, 0]):
    ax.add_patch(mpatches.FancyBboxPatch((0.5, y_pos), 3, 1.5, 
                 boxstyle="round,pad=0.1", facecolor='#4CAF50', alpha=0.3))
    ax.text(2, y_pos + 0.75, f'GPU {i}\n(own process)', ha='center', va='center', fontsize=9)

# Ring AllReduce arrows
ax.add_patch(mpatches.FancyBboxPatch((5, 2), 4.5, 4, 
             boxstyle="round,pad=0.1", facecolor='#4CAF50', alpha=0.1))
ax.text(7.25, 4, 'AllReduce\n(ring topology)\n\nEach GPU computes\nlocal gradients\n+ averages with peers', 
        ha='center', va='center', fontsize=9)

for y_pos in [6.75, 4.75, 2.75, 0.75]:
    ax.annotate('', xy=(5, 4), xytext=(3.8, y_pos),
                arrowprops=dict(arrowstyle='<->', color='green', lw=1.5))

plt.tight_layout()
plt.show()

## 3. AllReduce — How Gradient Sync Works

**AllReduce**: Every GPU starts with local gradients, ends with the average of all gradients.

**Ring AllReduce** (used in practice):
1. Arrange N GPUs in a ring
2. **Scatter-Reduce** phase: Each GPU sends a chunk to its neighbor, accumulates received chunks. After N-1 steps, each GPU has the full sum for one chunk.
3. **All-Gather** phase: Each GPU sends its complete chunk around the ring. After N-1 steps, all GPUs have the full averaged gradient.

Communication cost: `2(N-1)/N * data_size` — scales almost linearly with GPUs.

In [ ]:
def simulate_allreduce(local_gradients):
    """Simulate AllReduce: average gradients across GPUs."""
    n_gpus = len(local_gradients)
    
    print(f"=== AllReduce Simulation ({n_gpus} GPUs) ===")
    for i, grad in enumerate(local_gradients):
        print(f"  GPU {i} local gradient: {grad.tolist()}")
    
    # AllReduce = sum then divide
    global_grad = sum(local_gradients) / n_gpus
    
    print(f"\n  After AllReduce (average):")
    print(f"  All GPUs now have: {global_grad.tolist()}")
    return global_grad


# Simulate 4 GPUs, each with different local gradients
torch.manual_seed(42)
local_grads = [torch.randn(4) for _ in range(4)]
avg_grad = simulate_allreduce(local_grads)

In [ ]:
def simulate_ring_allreduce_scatter_reduce(local_gradients):
    """Step-by-step simulation of ring AllReduce (scatter-reduce phase)."""
    n = len(local_gradients)
    chunk_size = len(local_gradients[0]) // n
    
    # Split each GPU's gradient into n chunks
    chunks = []
    for grad in local_gradients:
        gpu_chunks = [grad[i*chunk_size:(i+1)*chunk_size].clone() for i in range(n)]
        chunks.append(gpu_chunks)
    
    print(f"Ring AllReduce Scatter-Reduce Phase ({n} GPUs, {n} chunks each):")
    
    # Scatter-Reduce: n-1 steps
    for step in range(n - 1):
        print(f"\n  Step {step + 1}:")
        new_chunks = [list(c) for c in chunks]  # copy
        for gpu_id in range(n):
            send_chunk_idx = (gpu_id - step) % n
            recv_from = (gpu_id - 1) % n
            recv_chunk_idx = (gpu_id - step) % n
            
            # GPU receives chunk from its left neighbor and accumulates
            new_chunks[gpu_id][recv_chunk_idx] = (
                chunks[gpu_id][recv_chunk_idx] + chunks[recv_from][recv_chunk_idx]
            )
            print(f"    GPU {gpu_id}: accumulate chunk {recv_chunk_idx} from GPU {recv_from}")
        
        chunks = new_chunks
    
    print(f"\n  After scatter-reduce: each GPU has the complete sum for one chunk.")
    return chunks


# Demo with 4 GPUs, gradient of size 4 (1 element per chunk)
torch.manual_seed(0)
grads_4 = [torch.tensor([1.0, 2.0, 3.0, 4.0]) * (i + 1) for i in range(4)]
for i, g in enumerate(grads_4):
    print(f"GPU {i}: {g.tolist()}")

result = simulate_ring_allreduce_scatter_reduce(grads_4)

## 4. DDP Training Script Pattern

This is the standard multi-GPU training pattern. It can't run in a notebook (needs `torchrun`), but the code is exactly what you'd write.

In [ ]:
# === DDP Training Script (save as train_ddp.py) ===
# Run with: torchrun --nproc_per_node=4 train_ddp.py

ddp_script = '''
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

def setup():
    """Initialize the distributed process group."""
    # torchrun sets these environment variables automatically
    dist.init_process_group(backend="nccl")  # "nccl" for GPU, "gloo" for CPU
    torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))

def cleanup():
    dist.destroy_process_group()

def main():
    setup()
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ["LOCAL_RANK"])
    device = torch.device(f"cuda:{local_rank}")
    
    # Model — wrapped in DDP
    model = nn.Linear(784, 10).to(device)
    model = DDP(model, device_ids=[local_rank])
    
    # Data — DistributedSampler partitions across GPUs
    dataset = torch.utils.data.TensorDataset(
        torch.randn(10000, 784), torch.randint(0, 10, (10000,))
    )
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    loader = DataLoader(dataset, batch_size=64, sampler=sampler, pin_memory=True)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(10):
        sampler.set_epoch(epoch)  # IMPORTANT: different shuffling each epoch
        model.train()
        
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            optimizer.zero_grad()
            loss.backward()      # DDP hooks trigger AllReduce here
            optimizer.step()
        
        if rank == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")
    
    # Save only on rank 0
    if rank == 0:
        torch.save(model.module.state_dict(), "model.pt")
    
    cleanup()

if __name__ == "__main__":
    main()
'''

print(ddp_script)

## 5. Model Parallelism & Pipeline Parallelism

### Model Parallelism (Naive)
Split model across GPUs. GPU 1 waits for GPU 0 → poor utilization (bubble problem).

### Pipeline Parallelism
Split model into stages AND split batch into micro-batches. While GPU 1 processes micro-batch 1, GPU 0 processes micro-batch 2.

In [ ]:
# Conceptual model parallelism (runs on CPU to show the pattern)

class ModelParallelDemo(nn.Module):
    """Demonstrates splitting a model across two 'devices'.
    In reality, replace 'cpu' with 'cuda:0', 'cuda:1'.
    """
    def __init__(self):
        super().__init__()
        # Stage 1 — would go on GPU 0
        self.stage1 = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
        )
        # Stage 2 — would go on GPU 1
        self.stage2 = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )
    
    def forward(self, x):
        # Stage 1 forward
        x = self.stage1(x)  # on GPU 0
        # x = x.to('cuda:1')  # transfer to GPU 1
        x = self.stage2(x)  # on GPU 1
        return x


model_mp = ModelParallelDemo()
x = torch.randn(8, 784)
out = model_mp(x)
print(f"Model parallel output shape: {out.shape}")
print(f"Stage 1 params: {sum(p.numel() for p in model_mp.stage1.parameters()):,}")
print(f"Stage 2 params: {sum(p.numel() for p in model_mp.stage2.parameters()):,}")

In [ ]:
# Visualize pipeline parallelism vs naive model parallelism

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Naive model parallelism — bubble problem
ax = axes[0]
ax.set_title('Naive Model Parallelism (Sequential)', fontweight='bold')
colors = plt.cm.Set2(np.linspace(0, 1, 4))

# GPU 0: forward then idle then backward
ax.broken_barh([(0, 2)], (1.5, 0.8), facecolors=colors[0], edgecolor='black', label='F micro-batch 1')
ax.broken_barh([(6, 2)], (1.5, 0.8), facecolors=colors[0], edgecolor='black', hatch='//')

# GPU 1: idle then forward then backward then idle
ax.broken_barh([(2, 2)], (0.5, 0.8), facecolors=colors[0], edgecolor='black')
ax.broken_barh([(4, 2)], (0.5, 0.8), facecolors=colors[0], edgecolor='black', hatch='//')

# Bubbles (idle time)
ax.broken_barh([(2, 4)], (1.5, 0.8), facecolors='white', edgecolor='gray', linestyle='--', alpha=0.5)
ax.broken_barh([(0, 2), (6, 2)], (0.5, 0.8), facecolors='white', edgecolor='gray', linestyle='--', alpha=0.5)

ax.set_yticks([1.9, 0.9])
ax.set_yticklabels(['GPU 0 (Stage 1)', 'GPU 1 (Stage 2)'])
ax.set_xlabel('Time')
ax.text(4, 1.9, 'IDLE', ha='center', va='center', fontsize=10, color='red')

# Pipeline parallelism — micro-batches
ax = axes[1]
ax.set_title('Pipeline Parallelism (4 micro-batches)', fontweight='bold')

# GPU 0 forward passes
for i in range(4):
    ax.broken_barh([(i, 0.9)], (1.5, 0.8), facecolors=colors[i], edgecolor='black')
    ax.text(i + 0.45, 1.9, f'F{i+1}', ha='center', va='center', fontsize=8)

# GPU 1 forward passes (shifted by 1)
for i in range(4):
    ax.broken_barh([(i + 1, 0.9)], (0.5, 0.8), facecolors=colors[i], edgecolor='black')
    ax.text(i + 1.45, 0.9, f'F{i+1}', ha='center', va='center', fontsize=8)

# GPU 0 backward passes
for i in range(4):
    ax.broken_barh([(5 + i, 0.9)], (1.5, 0.8), facecolors=colors[3-i], edgecolor='black', hatch='//')
    ax.text(5 + i + 0.45, 1.9, f'B{4-i}', ha='center', va='center', fontsize=8)

# GPU 1 backward passes
for i in range(4):
    ax.broken_barh([(5 + i, 0.9)], (0.5, 0.8), facecolors=colors[3-i], edgecolor='black', hatch='//')
    ax.text(5 + i + 0.45, 0.9, f'B{4-i}', ha='center', va='center', fontsize=8)

ax.set_yticks([1.9, 0.9])
ax.set_yticklabels(['GPU 0 (Stage 1)', 'GPU 1 (Stage 2)'])
ax.set_xlabel('Time')

plt.tight_layout()
plt.show()

print("Pipeline parallelism reduces bubbles by overlapping stages across micro-batches.")
print("Bubble fraction ≈ (num_stages - 1) / num_micro_batches")

## 6. Tensor Parallelism

Split individual layers across GPUs. For a linear layer `Y = XW`:

**Column parallel**: Split W along columns → each GPU computes part of Y → AllGather outputs  
**Row parallel**: Split W along rows → split X across GPUs → each GPU computes partial Y → AllReduce

Used in Megatron-LM for transformer attention and FFN layers.

In [ ]:
def simulate_tensor_parallel_column(X, W, n_gpus=2):
    """Simulate column-parallel linear: split W along output dim."""
    d_out = W.shape[1]
    chunk_size = d_out // n_gpus
    
    print(f"Column Parallel: X={X.shape}, W={W.shape}, {n_gpus} GPUs")
    
    partial_outputs = []
    for i in range(n_gpus):
        W_chunk = W[:, i*chunk_size:(i+1)*chunk_size]
        Y_chunk = X @ W_chunk  # Each GPU computes partial output
        partial_outputs.append(Y_chunk)
        print(f"  GPU {i}: W_chunk={W_chunk.shape}, Y_chunk={Y_chunk.shape}")
    
    # AllGather: concatenate partial outputs
    Y = torch.cat(partial_outputs, dim=-1)
    print(f"  After AllGather: Y={Y.shape}")
    
    # Verify correctness
    Y_full = X @ W
    assert torch.allclose(Y, Y_full, atol=1e-5)
    print("  Verified: matches full computation.")
    return Y


X = torch.randn(4, 8)
W = torch.randn(8, 16)
simulate_tensor_parallel_column(X, W, n_gpus=4)

## 7. FSDP — Fully Sharded Data Parallel

**Problem**: DDP replicates the full model on every GPU. For large models, optimizer states alone can be 12-20x the parameter count in memory.

**FSDP shards everything across GPUs**:

| What's sharded | ZeRO Stage | FSDP |
|---------------|------------|------|
| Optimizer states | ZeRO-1 | — |
| + Gradients | ZeRO-2 | `SHARD_GRAD_OP` |
| + Parameters | ZeRO-3 | `FULL_SHARD` |

**How it works**:
1. Parameters are sharded: each GPU stores 1/N of the params
2. Before forward: AllGather to reconstruct full parameters
3. After forward of a layer: discard unsharded params (free memory)
4. Before backward of a layer: AllGather again
5. After backward: ReduceScatter gradients, each GPU updates its shard

In [ ]:
def calculate_memory_per_gpu(param_count_billions, n_gpus, strategy):
    """Calculate memory per GPU for different strategies.
    
    Memory for training = params + gradients + optimizer states
    Adam optimizer: 2 states (mean, variance) per param = 8 bytes in FP32
    
    In mixed precision:
    - FP16 params: 2 bytes per param
    - FP32 master weights: 4 bytes per param
    - FP32 gradients: 4 bytes per param (accumulated in FP32)
    - Adam states: 8 bytes per param (FP32)
    Total ≈ 18 bytes per param (Φ)
    """
    P = param_count_billions * 1e9
    bytes_per_param = 18  # Mixed precision with Adam
    
    total_memory_gb = P * bytes_per_param / 1e9
    
    if strategy == 'DDP':
        # Full model replicated on each GPU
        per_gpu = total_memory_gb
    elif strategy == 'ZeRO-1':
        # Shard optimizer states only (8 bytes/param)
        per_gpu = P * (2 + 4 + 4) / 1e9 + P * 8 / n_gpus / 1e9
    elif strategy == 'ZeRO-2':
        # Shard optimizer states + gradients
        per_gpu = P * (2 + 4) / 1e9 + P * (4 + 8) / n_gpus / 1e9
    elif strategy == 'ZeRO-3' or strategy == 'FSDP':
        # Shard everything
        per_gpu = total_memory_gb / n_gpus
    
    return per_gpu


# Compare for a 7B parameter model on 8 GPUs
model_size = 7  # billion params
n_gpus = 8

print(f"Memory per GPU for {model_size}B param model on {n_gpus} GPUs:")
print(f"{'Strategy':<15} {'Memory/GPU':>12} {'Fits on 80GB?':>15}")
print("-" * 45)

for strategy in ['DDP', 'ZeRO-1', 'ZeRO-2', 'ZeRO-3']:
    mem = calculate_memory_per_gpu(model_size, n_gpus, strategy)
    fits = 'Yes' if mem < 80 else 'No'
    print(f"{strategy:<15} {mem:>10.1f} GB {fits:>14}")

In [ ]:
# Visualize memory breakdown
fig, ax = plt.subplots(figsize=(10, 5))

strategies = ['DDP', 'ZeRO-1', 'ZeRO-2', 'ZeRO-3/FSDP']
P = 7  # billion
N = 8  # GPUs

# Memory components (GB) per GPU
# [fp16_params, fp32_master, gradients, optimizer_states]
data = {
    'DDP':          [P*2, P*4, P*4, P*8],   # all replicated
    'ZeRO-1':       [P*2, P*4, P*4, P*8/N], # opt sharded
    'ZeRO-2':       [P*2, P*4, P*4/N, P*8/N],
    'ZeRO-3/FSDP':  [P*2/N, P*4/N, P*4/N, P*8/N],
}

x = np.arange(len(strategies))
width = 0.6
labels = ['FP16 Params', 'FP32 Master', 'Gradients', 'Optimizer States']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

bottom = np.zeros(4)
for i, (label, color) in enumerate(zip(labels, colors)):
    values = [data[s][i] for s in strategies]
    ax.bar(x, values, width, bottom=bottom, label=label, color=color)
    bottom += values

ax.axhline(y=80, color='red', linestyle='--', linewidth=2, label='80GB A100 limit')
ax.set_xticks(x)
ax.set_xticklabels(strategies)
ax.set_ylabel('Memory per GPU (GB)')
ax.set_title(f'Memory per GPU: {P}B Parameters, {N} GPUs')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8. Communication Overhead

| Collective | Data transferred per GPU | Use case |
|-----------|-------------------------|----------|
| AllReduce | 2 * (N-1)/N * D | DDP gradient sync |
| AllGather | (N-1)/N * D | FSDP parameter gathering |
| ReduceScatter | (N-1)/N * D | FSDP gradient sharding |

Where D = data size, N = number of GPUs.

**DDP overlaps communication with backward pass** — while computing gradients for layer L, it sends gradients for layer L+1. This hides most of the communication cost.

In [ ]:
# Communication overhead analysis
def comm_overhead(param_gb, n_gpus, bandwidth_gbps=300):
    """Estimate communication time for AllReduce.
    
    Args:
        param_gb: model parameters in GB
        n_gpus: number of GPUs
        bandwidth_gbps: interconnect bandwidth (NVLink: ~300 GB/s, PCIe: ~32 GB/s)
    """
    # AllReduce transfers 2*(N-1)/N * data_size
    data_transferred = 2 * (n_gpus - 1) / n_gpus * param_gb
    time_ms = data_transferred / bandwidth_gbps * 1000
    return time_ms, data_transferred


print("AllReduce Communication Time (per step):")
print(f"{'Model':>10} {'GPUs':>5} {'Interconnect':>15} {'Data Xfer':>12} {'Time':>10}")
print("-" * 55)

for model_name, param_gb in [('350M', 0.7), ('7B', 14), ('70B', 140)]:
    for n_gpus in [4, 8]:
        for bw_name, bw in [('NVLink', 300), ('PCIe', 32)]:
            time_ms, data_gb = comm_overhead(param_gb, n_gpus, bw)
            print(f"{model_name:>10} {n_gpus:>5} {bw_name:>15} {data_gb:>10.1f} GB {time_ms:>8.1f} ms")

## Interview Questions & Answers

---

**Q: DDP vs DP?**

- DP: single process, GIL bottleneck, uneven GPU memory (GPU 0 does all gather/scatter). Never use.
- DDP: one process per GPU, AllReduce for gradient sync, overlaps comm with compute. Always use DDP.

---

**Q: How does gradient synchronization work in DDP?**

1. Each GPU computes local gradients on its data shard
2. DDP hooks trigger AllReduce after each backward pass
3. Ring AllReduce: `2(N-1)/N * D` bytes transferred
4. Communication overlaps with backward computation (bucketing)
5. After sync, all GPUs have identical averaged gradients
6. Each GPU applies the same optimizer step → parameters stay in sync

---

**Q: When FSDP vs DDP?**

- **DDP**: Model fits comfortably on one GPU. Simpler, less communication.
- **FSDP (SHARD_GRAD_OP = ZeRO-2)**: Model barely fits on one GPU. Shard gradients + optimizer states.
- **FSDP (FULL_SHARD = ZeRO-3)**: Model doesn't fit on one GPU. Shard everything. More communication (AllGather before each forward/backward), but enables much larger models.

---

**Q: Communication overhead — how to minimize?**

1. Use fast interconnect (NVLink > PCIe)
2. DDP automatically overlaps comm with backward
3. Gradient compression (quantize before AllReduce)
4. Gradient accumulation (reduce sync frequency)
5. Increase compute-to-communication ratio (larger batch per GPU)

---

**Q: DeepSpeed ZeRO stages?**

| Stage | What's sharded | Memory saving | Comm overhead |
|-------|---------------|---------------|---------------|
| ZeRO-1 | Optimizer states | ~4x | Same as DDP |
| ZeRO-2 | + Gradients | ~8x | Same as DDP |
| ZeRO-3 | + Parameters | ~N*x (linear with GPUs) | 1.5x DDP |

ZeRO-3 = FSDP FULL_SHARD. The extra communication in ZeRO-3 comes from AllGather before forward and backward passes.

## Quick Reference

```bash
# Launch DDP training
torchrun --nproc_per_node=4 train.py

# Multi-node
torchrun --nproc_per_node=8 --nnodes=2 --node_rank=0 \
    --master_addr=10.0.0.1 --master_port=29500 train.py
```

```python
# FSDP pattern
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
model = FSDP(model, sharding_strategy=ShardingStrategy.FULL_SHARD)
```

**Typical recipe for LLM training**:
- 1-10B params: FSDP or ZeRO-2/3 on 8-64 GPUs
- 10-100B params: 3D parallelism (TP + PP + DP) with Megatron/DeepSpeed
- 100B+ params: 3D parallelism + ZeRO + offloading